# Customer spending by decile: Lorenz curve

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Lorenz Curve and Gini Coefficient** · a worked example from the StatNex teaching guide

**The story.** An online shop ranked its customers by yearly spending and split them into ten equal groups (deciles). For each decile it reports the average annual spend per customer.

**The question.** How concentrated is revenue among the best customers?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `11_chart/lorenz_curve__deciles.csv` (10 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Customer decile,Annual spend per customer ($)
D1,12.0
D2,25.0
D3,41.0
D4,60.0
D5,84.0
D6,118.0
D7,170.0
D8,255.0
D9,420.0
D10,1310.0

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
# One value per unit (or per class, each counted once). Text and blank cells are
# skipped.
inputs = {}
for name in ["Annual spend per customer ($)"]:
    values = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy()
    inputs[name] = (values, np.ones(len(values)))

## Lorenz curve and concentration indices

In [ ]:
# Units sorted from smallest to largest. The curve plots the cumulative share of the
# population (p) against the cumulative share of the total they hold (L). SciPy and
# statsmodels have no Lorenz or Gini function, so both are written out.
#   Gini    = twice the area between the diagonal and the curve (trapezium rule,
#             exact because the curve is made of straight pieces): 0 when everyone
#             holds the same, near 1 when one unit holds everything. This is the
#             plain Gini, without the n/(n - 1) "unbiased" factor some packages add.
#   HHI     = sum of squared shares; 1/n when everything is equal
#   Entropy = -sum(share * log(share)); log(n) when everything is equal
#   Pietra  = the widest vertical gap between the diagonal and the curve: the share
#             of the total that would have to move to make everyone equal
# With grouped data the HHI and the entropy take each class's total as one share
# (how the total splits across the classes), while the curve and the Gini count
# every unit. A curve built from class or decile averages ignores the spread inside
# each class, so its Gini understates the Gini across the individual units.


def lorenz(values, weights):
    order = np.argsort(values, kind="stable")
    values, weights = values[order], weights[order]
    amount = (values * weights).sum()
    p = np.concatenate([[0.0], np.cumsum(weights) / weights.sum()])
    curve = np.concatenate([[0.0], np.cumsum(values * weights) / amount])
    share = values * weights / amount
    positive = share[share > 0]
    return {
        "p": p, "curve": curve, "n": len(values),
        "gini": 1 - ((p[1:] - p[:-1]) * (curve[1:] + curve[:-1])).sum(),
        "hhi": (share**2).sum(),
        "entropy": -(positive * np.log(positive)).sum(),
        "pietra": (p - curve).max(),
    }


results = {name: lorenz(*pair) for name, pair in inputs.items()}
indices = pd.DataFrame({name: [r["gini"], r["hhi"], r["hhi"] * 10000, r["entropy"]]
                        for name, r in results.items()},
                       index=["Gini coefficient", "HHI (ratio)", "HHI (%)", "Entropy"])
print(indices)
extra = pd.DataFrame({name: [r["pietra"], 1 / r["n"], np.log(r["n"])]
                      for name, r in results.items()},
                     index=["Pietra index", "HHI lower bound (1/n)", "Entropy upper bound (log n)"])
print()
print(extra)

## Points of the curve

In [ ]:
# Every series on one grid of p, interpolating where a series has no point of its own.
grid = sorted({value for r in results.values() for value in r["p"]})
chart = pd.DataFrame({"p": grid, **{name: np.interp(grid, r["p"], r["curve"])
                                    for name, r in results.items()}})
print()
print(chart.to_string(index=False))

## Lorenz curve

In [ ]:
Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Perfect equality")
for name, r in results.items():
    ax.plot(r["p"], r["curve"], marker="o", label=name)
ax.set_xlabel("Cumulative share of the population")
ax.set_ylabel("Cumulative share of the total")
ax.legend()
fig.savefig("figures/lorenz.png", dpi=120, bbox_inches="tight")

## What the result says

The Lorenz curve puts customers in order from the lowest spender to the highest and shows the share of revenue they bring in together. The bottom half of customers (0.5 on the horizontal axis) bring in only 0.089 of revenue, about 9%. The bottom nine tenths bring in 0.475, so the top tenth alone brings in about half.

The Gini coefficient sums this up in one number, from 0 (every customer spends the same) to 1 (one customer spends everything). Here it is 0.64, which is high.

Revenue is heavily concentrated. That makes the best customers precious, and it makes revenue fragile if some of them leave.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.